In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
import glob

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

#Create the spark session
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

#Path to the folder with all the files
files = glob.glob(os.path.join(r"C:\Users\mekay\Downloads\MIT805 Data", "*.parquet"))

#Import failed a few times due to varying datatypes for the same column: passenger_count
#Decided to just convert all of them to doubles

dfs = [] #list for all our dataframes that we will eventually union
for f in files:
    d = spark.read.parquet(f)
    d = d.withColumn("passenger_count", col("passenger_count").cast("double"))
    dfs.append(d)

#Start with our first dataframe and we will append to this for our final dataframe
df = dfs[0]

#Go through the list and append them
for d in dfs[1:]:
    df = df.unionByName(d, allowMissingColumns=True)

C:\Users\mekay\anaconda3\envs\Mekayl\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Data pre-processing 

In [2]:
"Number of rows before cleaning"
df.count()

429881159

In [3]:
"Datatypes of the columns"
df.dtypes

[('VendorID', 'bigint'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'bigint'),
 ('DOLocationID', 'bigint'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [4]:
##Creating our query table

df.createOrReplaceTempView("taxi_data")

In [5]:
## Min and Max of each variable

ranges_df = spark.sql("""
    SELECT 'passenger_count' AS Variable, ROUND(MIN(passenger_count),2) AS Lower, ROUND(AVG(passenger_count),2) AS Average, ROUND(MAX(passenger_count),2) AS Highest FROM taxi_data
    UNION ALL
    SELECT 'trip_distance', ROUND(MIN(trip_distance),2), ROUND(AVG(trip_distance),2), ROUND(MAX(trip_distance),2) FROM taxi_data
    UNION ALL
    SELECT 'fare_amount', ROUND(MIN(fare_amount),2), ROUND(AVG(fare_amount),2), ROUND(MAX(fare_amount),2) FROM taxi_data
    UNION ALL
    SELECT 'extra', ROUND(MIN(extra),2), ROUND(AVG(extra),2), ROUND(MAX(extra),2) FROM taxi_data
    UNION ALL
    SELECT 'mta_tax', ROUND(MIN(mta_tax),2), ROUND(AVG(mta_tax),2), ROUND(MAX(mta_tax),2) FROM taxi_data
    UNION ALL
    SELECT 'tip_amount', ROUND(MIN(tip_amount),2), ROUND(AVG(tip_amount),2), ROUND(MAX(tip_amount),2) FROM taxi_data
    UNION ALL
    SELECT 'tolls_amount', ROUND(MIN(tolls_amount),2), ROUND(AVG(tolls_amount),2), ROUND(MAX(tolls_amount),2) FROM taxi_data
    UNION ALL
    SELECT 'improvement_surcharge', ROUND(MIN(improvement_surcharge),2), ROUND(AVG(improvement_surcharge),2), ROUND(MAX(improvement_surcharge),2) FROM taxi_data
    UNION ALL
    SELECT 'total_amount', ROUND(MIN(total_amount),2), ROUND(AVG(total_amount),2), ROUND(MAX(total_amount),2) FROM taxi_data
    UNION ALL
    SELECT 'congestion_surcharge', ROUND(MIN(congestion_surcharge),2), ROUND(AVG(congestion_surcharge),2), ROUND(MAX(congestion_surcharge),2) FROM taxi_data
    UNION ALL
    SELECT 'airport_fee', ROUND(MIN(airport_fee),2), ROUND(AVG(airport_fee),2), ROUND(MAX(airport_fee),2) FROM taxi_data
""")

ranges_pdf = ranges_df.toPandas()
ranges_pdf

C:\Users\mekay\anaconda3\envs\Mekayl\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,Variable,Lower,Average,Highest
0,passenger_count,0.000000e+00,1.47,1.920000e+02
1,trip_distance,-3.726453e+04,4.42,3.986086e+05
2,fare_amount,-1.333914e+08,15.04,9.983100e+05
3,extra,-8.000000e+01,0.98,5.000008e+05
4,mta_tax,-2.174000e+01,0.49,5.000005e+05
5,tip_amount,-4.932200e+02,2.91,1.333914e+08
6,tolls_amount,-1.481700e+02,0.44,3.288000e+03
7,improvement_surcharge,-1.000000e+00,0.53,4.000300e+03
8,total_amount,-2.567800e+03,21.76,1.084772e+06
9,congestion_surcharge,-2.500000e+00,2.23,4.500000e+00


In [6]:
"Number of Nulls"

null_counts = spark.sql("""
    SELECT
        SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS VendorID,
        SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS tpep_pickup_datetime,
        SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS tpep_dropoff_datetime,
        SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count,
        SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance,
        SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) AS RatecodeID,
        SUM(CASE WHEN store_and_fwd_flag IS NULL THEN 1 ELSE 0 END) AS store_and_fwd_flag,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS PULocationID,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS DOLocationID,
        SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type,
        SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS fare_amount,
        SUM(CASE WHEN extra IS NULL THEN 1 ELSE 0 END) AS extra,
        SUM(CASE WHEN mta_tax IS NULL THEN 1 ELSE 0 END) AS mta_tax,
        SUM(CASE WHEN tip_amount IS NULL THEN 1 ELSE 0 END) AS tip_amount,
        SUM(CASE WHEN tolls_amount IS NULL THEN 1 ELSE 0 END) AS tolls_amount,
        SUM(CASE WHEN improvement_surcharge IS NULL THEN 1 ELSE 0 END) AS improvement_surcharge,
        SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount,
        SUM(CASE WHEN congestion_surcharge IS NULL THEN 1 ELSE 0 END) AS congestion_surcharge,
        SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END) AS airport_fee,
        SUM(CASE WHEN cbd_congestion_fee IS NULL THEN 1 ELSE 0 END) AS cbd_congestion_fee
    FROM taxi_data
""")

null_counts.toPandas().T

C:\Users\mekay\anaconda3\envs\Mekayl\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,25984554
trip_distance,0
RatecodeID,25984554
store_and_fwd_flag,25984554
PULocationID,0
DOLocationID,0
payment_type,0


In [7]:
"Removing all NULL values and bounding other variable so that our data is realistic"
    
    
cleaned_result = spark.sql("""
        SELECT distinct 
        CAST(VendorID AS DOUBLE) AS VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        CAST(passenger_count AS DOUBLE) AS passenger_count,
        CAST(trip_distance AS DOUBLE) AS trip_distance,
        CAST(RatecodeID AS DOUBLE) AS RatecodeID,
        CAST(PULocationID AS DOUBLE) AS PULocationID,
        CAST(DOLocationID AS DOUBLE) AS DOLocationID,
        CAST(payment_type AS DOUBLE) AS payment_type,
        CAST(fare_amount AS DOUBLE) AS fare_amount,
        CAST(tip_amount AS DOUBLE) AS tip_amount,
        CAST(total_amount AS DOUBLE) AS total_amount,
        CAST(((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS DOUBLE) AS trip_duration,
        CAST(trip_distance/((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS DOUBLE) AS average_trip_speed
       
        FROM taxi_data
       
        where 
        (passenger_count > 0 and passenger_count <= 4)
        and (trip_distance > 0 and trip_distance < 20)
        and (fare_amount > 0 and fare_amount <100)
        and (tip_amount >= 0)
        and (total_amount > 0 and total_amount < 2000 )
        and( 
        ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) > 0 
        and
        ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) < 3 
        )
        and (trip_distance/((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) > 0
        and trip_distance/((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) < 55)
        and (year(tpep_pickup_datetime) >=2014 and year(tpep_pickup_datetime) <=2025)
    """)

In [8]:
"Number of rows after cleaning"
cleaned_result.count()

355237231

In [9]:
"Taking a 20% sample "
sampled_result = cleaned_result.sample(withReplacement=False, fraction=0.2, seed=27)

In [10]:
sampled_result.coalesce(1).write.mode("overwrite").parquet(r"C:\Users\mekay\Downloads\MIT805 Data\Final File")